# 03. Serving the Model

## 📚 Learning Objectives

By the end of this notebook you will be able to:

- **Serialize** a trained scikit-learn model with `joblib`, reload it, and prove the copy predicts exactly what the original did.
- **Serve** it behind an HTTP endpoint with FastAPI: POST one row of measurements, get a prediction and a probability back.
- **Defend** the endpoint: send it malformed rows and watch each one get rejected *before* it reaches the model, then say what validation cannot catch.
- Explain, with a demonstration, why a model file from a stranger is executable code.

## 🔗 Where this fits

**Builds on:** `01_ai_learning_models.ipynb` in this unit, whose logistic regression on the Wisconsin biopsies is the model we serve here; Course 01 (AIAT 111) Unit 1, `enrichment/E1_agent_loop_without_a_model.ipynb`, where a validator rejected a bad tool call before execution. This is the same idea at an HTTP boundary.

**Used later in:** Course 11 (AIAT 125) Unit 2, `examples/02_fastapi_deployment.ipynb` (a real server process, not an in-process client), `examples/04_saving_loading_models_pickle_onnx.ipynb` (formats that are not pickle) and `examples/03_model_versioning.ipynb` (what the fingerprint in this lesson's responses is for).

This lesson delivers the unit's official bullet *"Deploying AI Models (serialization, API development, production)"* and its practical twins *"Serializing and saving models for deployment"* and *"Building simple APIs for model serving (Flask, FastAPI)"*.

## The Story: a model in a notebook variable is not a product

Lesson 01 ended with `model.predict(...)` working in a cell. Close the notebook and it is gone. Nobody outside this kernel can ask it anything, and nobody can send it a row that was typed by a person in a hurry. Getting from that variable to something a colleague can call is three hand movements, and this notebook does each one:

1. **Save** the model to a file, reload it, and check the copy is faithful.
2. **Serve** it: an HTTP endpoint that takes one row as JSON and returns the prediction.
3. **Defend** it: decide what happens when the row is wrong, and make sure a wrong row never reaches the model.

The model is the one lesson 01 built: logistic regression on two real measurements (mean radius and mean texture of the cell nuclei) from the 569 Wisconsin Diagnostic Breast Cancer biopsies, rebuilt here with the same split and the same seed. Everything this notebook writes goes to a throw-away folder that is deleted in the last cell.

## 📥 Inputs & 📤 Outputs

**Inputs**

- `sklearn.datasets.load_breast_cancer`: the 569 real biopsies, bundled with scikit-learn (no network).
- `joblib`, `fastapi`, `pydantic`, `httpx` (for FastAPI's in-process test client): all in the diploma's `requirements.txt`.
- Kernel: **`ai-diploma`**.

**Outputs**

- A model file in a temporary folder (deleted at the end), its size and its fingerprint.
- Printed HTTP responses: one good prediction, five rejected requests, one accepted-but-suspicious request.
- No figure: nothing in this lesson is a shape.

In [1]:
import numpy as np
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score

cancer = load_breast_cancer()
FEATURES = ["mean_radius", "mean_texture"]        # the same two of the 30 real measurements that lesson 01 used
X = cancer.data[:, [0, 1]]
y = (cancer.target == 0).astype(int)              # 1 = malignant, the thing we hunt for, as in lesson 01
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = LogisticRegression(random_state=42, max_iter=1000).fit(X_train, y_train)

pred = model.predict(X_test)
print(f"{len(X)} real biopsies (Wisconsin Diagnostic Breast Cancer), {len(X_test)} held out")
print(f"held-out accuracy {accuracy_score(y_test, pred):.1%}; recall on malignant {recall_score(y_test, pred):.1%}")
TRAIN_RANGE = {f: (round(float(X_train[:, i].min()), 2), round(float(X_train[:, i].max()), 2))
               for i, f in enumerate(FEATURES)}
print("range of values the model saw in training:", TRAIN_RANGE)

569 real biopsies (Wisconsin Diagnostic Breast Cancer), 171 held out
held-out accuracy 90.6%; recall on malignant 82.5%
range of values the model saw in training: {'mean_radius': (7.69, 28.11), 'mean_texture': (9.71, 39.28)}


## Part 1: Save it, reload it, prove the copy is faithful

`joblib.dump` writes the Python object to a file; `joblib.load` rebuilds it. We save a small **bundle** rather than the bare estimator: the feature order and the training range travel with the model, because a service that has to *guess* which column is which is a service that will one day guess wrong.

In [2]:
import hashlib
import tempfile
from pathlib import Path
import joblib

SCRATCH = Path(tempfile.mkdtemp(prefix="unit5_serving_"))    # a throw-away folder outside the repository
MODEL_PATH = SCRATCH / "biopsy_model.joblib"
joblib.dump({"model": model, "features": FEATURES, "train_range": TRAIN_RANGE, "sklearn": sklearn.__version__},
            MODEL_PATH)
MODEL_SHA = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()[:12]   # a fingerprint: any change to the file changes it
print(f"wrote {MODEL_PATH.name}: {MODEL_PATH.stat().st_size:,} bytes, fingerprint {MODEL_SHA}")

bundle = joblib.load(MODEL_PATH)
reloaded = bundle["model"]
print("reloaded model predicts exactly what the original does on every held-out row:",
      bool(np.array_equal(reloaded.predict(X_test), model.predict(X_test))))
print("the file also carries:", {k: v for k, v in bundle.items() if k != "model"})

wrote biopsy_model.joblib: 998 bytes, fingerprint fe9110b354ba
reloaded model predicts exactly what the original does on every held-out row: True
the file also carries: {'features': ['mean_radius', 'mean_texture'], 'train_range': {'mean_radius': (7.69, 28.11), 'mean_texture': (9.71, 39.28)}, 'sklearn': '1.8.0'}


### The check scikit-learn does for you, and what it looks like when it fires

scikit-learn stamps every estimator it pickles with the version that wrote it, and compares that stamp with the version that reads it. There is no supported way to load a model across versions, so the stamp is your early warning. To see the warning without installing an old scikit-learn, the next cell **forges the stamp** at save time. A real mismatch produces the same message.

In [3]:
import warnings
from unittest.mock import patch
from sklearn.exceptions import InconsistentVersionWarning

OLD_PATH = SCRATCH / "biopsy_model_from_old_sklearn.joblib"
with patch("sklearn.base.__version__", "1.2.0"):    # forged: this file now claims scikit-learn 1.2.0 wrote it
    joblib.dump(model, OLD_PATH)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    joblib.load(OLD_PATH)
for w in caught:
    if issubclass(w.category, InconsistentVersionWarning):
        print(f"{w.category.__name__}: {w.message}")

https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations


## Part 2: Serve it

FastAPI turns a Python function into an HTTP endpoint. The **pydantic model** `Biopsy` is the contract: what a request must contain, what type each field is, what values are allowed. Anything that does not match is rejected with status **422** before our function runs.

There is no server process here. `TestClient` calls the app in this kernel, with no port and no network, so the notebook works the same on Colab, Windows and a locked-down classroom machine. Course 11 runs the same app under a real server.

In [4]:
import warnings
from fastapi import FastAPI
from pydantic import BaseModel, ConfigDict, Field
warnings.filterwarnings("ignore", message="Using `httpx` with `starlette.testclient`")   # packaging notice in some starlette versions; not part of the lesson
from fastapi.testclient import TestClient

BUNDLE = joblib.load(MODEL_PATH)      # loaded ONCE when the service starts, not once per request

class Biopsy(BaseModel):
    model_config = ConfigDict(extra="forbid")     # an unknown field is a bug in the caller, not something to ignore
    mean_radius: float = Field(gt=0, description="mean radius of the cell nuclei")
    mean_texture: float = Field(gt=0, description="mean texture of the cell nuclei")

app = FastAPI(title="Biopsy classifier", version=MODEL_SHA)
CALLS_TO_MODEL = 0

@app.get("/health")
def health():
    return {"status": "ok", "model_fingerprint": MODEL_SHA, "sklearn": BUNDLE["sklearn"], "features": BUNDLE["features"]}

@app.post("/predict")
def predict(row: Biopsy):
    global CALLS_TO_MODEL
    CALLS_TO_MODEL += 1
    x = np.array([[row.mean_radius, row.mean_texture]])
    p_malignant = float(BUNDLE["model"].predict_proba(x)[0, 1])
    inside = all(lo <= getattr(row, f) <= hi for f, (lo, hi) in BUNDLE["train_range"].items())
    return {"prediction": "malignant" if p_malignant >= 0.5 else "benign",
            "p_malignant": round(p_malignant, 4),
            "inside_training_range": inside,        # the schema checked the shape; this is the first check of meaning
            "model_fingerprint": MODEL_SHA}

client = TestClient(app)
print(client.get("/health").json())

{'status': 'ok', 'model_fingerprint': 'fe9110b354ba', 'sklearn': '1.8.0', 'features': ['mean_radius', 'mean_texture']}


In [5]:
i = 0
row = {"mean_radius": round(float(X_test[i, 0]), 2), "mean_texture": round(float(X_test[i, 1]), 2)}
response = client.post("/predict", json=row)
print("sent:    ", row)
print("status:  ", response.status_code)
print("received:", response.json())
print("pathology result for that biopsy:", "malignant" if y_test[i] == 1 else "benign")

# the service and the notebook must agree, row for row; if they ever disagree, the file is not the model you think it is
agree = all(client.post("/predict", json={"mean_radius": float(r), "mean_texture": float(t)}).json()["p_malignant"]
            == round(float(model.predict_proba([[r, t]])[0, 1]), 4) for r, t in X_test[:20])
print(f"service agrees with the notebook's own model on the first 20 held-out rows: {agree}")

sent:     {'mean_radius': 12.47, 'mean_texture': 18.6}
status:   200
received: {'prediction': 'benign', 'p_malignant': 0.0831, 'inside_training_range': True, 'model_fingerprint': 'fe9110b354ba'}
pathology result for that biopsy: benign
service agrees with the notebook's own model on the first 20 held-out rows: True


## Part 3: Break it

Five requests a real caller will send sooner or later. Watch two things: the status code and the counter of how many times the model was actually called.

In [6]:
bad_requests = {
    "a word where a number should be":    {"mean_radius": "fourteen", "mean_texture": 20.1},
    "a missing field":                    {"mean_radius": 14.2},
    "an extra field the model never saw": {"mean_radius": 14.2, "mean_texture": 20.1, "mean_area": 650.0},
    "a negative radius":                  {"mean_radius": -14.2, "mean_texture": 20.1},
    "a whole CSV line as text":           "14.2,20.1",
}
before = CALLS_TO_MODEL
for label, body in bad_requests.items():
    r = client.post("/predict", json=body)
    detail = r.json()["detail"][0]
    where = ".".join(str(p) for p in detail["loc"] if p != "body") or "body"
    print(f"{r.status_code}  {label:<36} -> {detail['msg']}  [{where}]")
print(f"\nrequests sent: {len(bad_requests)}; times the model was called: {CALLS_TO_MODEL - before}")

422  a word where a number should be      -> Input should be a valid number, unable to parse string as a number  [mean_radius]
422  a missing field                      -> Field required  [mean_texture]
422  an extra field the model never saw   -> Extra inputs are not permitted  [mean_area]
422  a negative radius                    -> Input should be greater than 0  [mean_radius]
422  a whole CSV line as text             -> Input should be a valid dictionary or object to extract fields from  [body]

requests sent: 5; times the model was called: 0


Every one of the five was stopped at the door. Now the request that **passes** the door: correct fields, correct types, positive numbers, and a radius no biopsy in the training set came close to.

In [7]:
response = client.post("/predict", json={"mean_radius": 45.0, "mean_texture": 20.1})
print(response.status_code, response.json())
print("largest mean radius the model saw in training:", BUNDLE["train_range"]["mean_radius"][1])

200 {'prediction': 'malignant', 'p_malignant': 1.0, 'inside_training_range': False, 'model_fingerprint': 'fe9110b354ba'}
largest mean radius the model saw in training: 28.11


The schema validates **shape**, not **meaning**. A radius of 45 is a valid float, so the model was called and answered with high confidence about a region of the input it has never seen. The response flags it; whether to refuse, warn or answer silently is a decision for the discussion below, not for pydantic.

## Part 4: What `joblib.load` actually does

`joblib.load` is `pickle` underneath, and pickle does not read data: it **runs a small program** that rebuilds objects. Any object can say how it should be rebuilt, and the answer is a function call. The next cell saves an object that is not a model, and loads it.

In [8]:
class LooksLikeAModel:
    def __reduce__(self):      # pickle asks every object "how do I rebuild you?"; the answer is a function and its arguments
        return (print, ("    <- this line was printed by joblib.load, before you did anything with the object",))

NOT_A_MODEL = SCRATCH / "not_a_model.joblib"
joblib.dump(LooksLikeAModel(), NOT_A_MODEL)
print("loading a file that claims to be a model...")
obj = joblib.load(NOT_A_MODEL)
print("what came back:", obj)

loading a file that claims to be a model...
    <- this line was printed by joblib.load, before you did anything with the object
what came back: None


`print` is harmless. Replace it with `os.system` and the file opens a shell on whatever machine loads it. That is why the scikit-learn documentation says pickle "should only be used if the artifact, i.e. the pickle-file, is coming from a trusted and verified source. You should never load a pickle file from an untrusted source, similarly to how you should never execute code from an untrusted source."

## 💬 Discuss

1. The endpoint answered the radius-45 request with `inside_training_range: false` and a prediction anyway. Three policies are possible: answer silently, answer with the flag (what we did), or refuse with a 4xx. Each one moves a decision onto somebody: the caller, the doctor, or the team that owns the model. Who should hold it, and what would each policy cost on the day the model is wrong?
2. `extra="forbid"` rejected a row that carried one field too many. Accepting it would be friendlier to callers and would break nothing today. Name a situation where that friendliness costs you, and what you would log if you chose to accept.
3. Every response carries the file fingerprint. On the day you retrain, the fingerprint changes. List who has to notice, and what goes wrong if two different model files ever answer with the same fingerprint.

## ⚠️ Where this breaks

- **A model file is a program, and people have used that.** On 27 February 2024 JFrog's security research team reported around 100 models on the Hugging Face hub with real, harmful payloads. In the case they detailed, a PyTorch model file used the pickle `__reduce__` mechanism, the same one the cell above used to call `print`, to open a reverse shell to an outside server when the model was loaded. Loading a `.joblib`, `.pkl` or pickle-based `.pt` file from a source you do not control is running that source's code on your machine. Course 11 Unit 2 shows formats built so that loading cannot execute anything.
- **Versions.** The scikit-learn documentation is blunt: "there are no supported ways to load a model trained with a different version of scikit-learn". The warning you triggered above is the only guard rail. Pin the version in the service, and store it beside the model, as the bundle here does.
- **`TestClient` is not a server.** No port, no concurrency, no authentication, no timeouts, no logging, no monitoring. Everything that fails in production fails in those six places, and none of them exists here. This lesson is the contract and the checks; Course 11 is the server.
- **Validation checks shape, not meaning.** The radius-45 request passed every check and got a confident answer. Ranges are a first defence, not a last one, and they are only as good as the training data you computed them from.
- **The model itself.** Two features, 171 held-out rows, and lesson 01 showed it misses malignant tumours at the 0.5 threshold. Serving a model does not make it better; it makes its mistakes reachable from a URL.

## ✅ Summary

- `joblib.dump` / `joblib.load` save and restore a model; check the copy against the original, and save the feature order and version with it.
- A pydantic schema is the contract at the door: wrong type, missing field, extra field and impossible value are all rejected with 422 before the model is called.
- Validation cannot judge meaning. An in-range check is the first defence and a policy decision is the second.
- A pickle-based file is executable. Load only what you trust; prefer formats that cannot run code.

## 📚 References

- scikit-learn developers. *Model persistence.* https://scikit-learn.org/stable/model_persistence.html (the version stamp, the security warning, and the alternatives: skops, ONNX)
- JFrog Security Research (27 February 2024). *Data Scientists Targeted by Malicious Hugging Face ML Models with Silent Backdoor.* https://jfrog.com/blog/data-scientists-targeted-by-malicious-hugging-face-ml-models-with-silent-backdoor/ (the named case in "Where this breaks")

In [9]:
import shutil
shutil.rmtree(SCRATCH)      # leave the machine as we found it
print("deleted", SCRATCH)

deleted /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/unit5_serving_ot2f2ubl
